In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import torch

# GPUが利用可能か確認
if torch.cuda.is_available():
    print("GPU is available!")

    # GPUデバイスの数を取得
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs: {num_gpus}")

    for i in range(num_gpus):
        # GPUデバイス名を取得
        gpu_name = torch.cuda.get_device_name(i)
        print(f"GPU {i}: {gpu_name}")

        # GPUのプロパティを取得 (CUDA capabilityなど)
        gpu_properties = torch.cuda.get_device_properties(i)
        print(f"GPU {i} properties: {gpu_properties}")

        # 現在のGPUデバイスを設定 (複数GPUがある場合)
        # torch.cuda.set_device(i)

        # 現在のGPUデバイスのインデックスを取得
        current_device = torch.cuda.current_device()
        print(f"Current device index: {current_device}")

else:
    print("GPU is not available.")

GPU is available!
Number of GPUs: 1
GPU 0: NVIDIA GeForce GTX 1660 SUPER
GPU 0 properties: _CudaDeviceProperties(name='NVIDIA GeForce GTX 1660 SUPER', major=7, minor=5, total_memory=6143MB, multi_processor_count=22, uuid=61975011-93d9-ca37-1436-007b0f5d96b2, L2_cache_size=1MB)
Current device index: 0


In [3]:
import sys
import argparse
import os.path
import os
import random
import time
import datetime
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
from PIL import Image
from visdom import Visdom

In [4]:
class AlignedDataset(Dataset):
    def __init__(self, config):
        self.config = config
        self.dir = os.path.join(config.dataroot, config.phase)  # データセットのルートディレクトリ
        self.image_pairs = []  # 画像ペアのファイルパスを格納するリスト

        for folder_name in os.listdir(self.dir):
            folder_path = os.path.join(self.dir, folder_name)
            if os.path.isdir(folder_path):
                # フォルダ内のテンソル画像とjpg画像を検索
                tensor_files = [f for f in os.listdir(folder_path) if f.endswith('.pt')]
                jpg_files = [f for f in os.listdir(folder_path) if f.endswith('.jpg')]

                # テンソル画像とjpg画像が1枚ずつ存在する場合、ペアとして追加
                if len(tensor_files) == 1 and len(jpg_files) == 1:
                    tensor_path = os.path.join(folder_path, tensor_files[0])
                    jpg_path = os.path.join(folder_path, jpg_files[0])
                    self.image_pairs.append((tensor_path, jpg_path))

    def __getitem__(self, index):
        tensor_path, jpg_path = self.image_pairs[index]  # 画像ペアのファイルパスを取得
        tensor_image = torch.load(tensor_path, weights_only=True).float() # テンソル画像を読み込み、float型に変換

        x_max = 508 - 256
        y_max = 286 - 256
        x = random.randint(0, np.maximum(0, x_max))
        y = random.randint(0, np.maximum(0, y_max))
        tensor_image = tensor_image[:,x:(x+256),y:(y+256)]

        # 左右反転を行うかどうかをランダムに決定（tensor_imageとjpg_imageで共有）
        # do_flip = random.random() < 0.5

        # テンソル画像の反転
        # if do_flip:
            # tensor_image = torch.flip(tensor_image, dims=[2])
        
        # jpg画像を読み込み、テンソルに変換
        jpg_image = Image.open(jpg_path).convert('RGB')
        jpg_transform = transforms.Compose([
            # transforms.RandomHorizontalFlip(p=1.0 if do_flip else 0.0), # do_flipの値に応じて反転
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        jpg_image = jpg_transform(jpg_image)
        jpg_image = jpg_image[:,x:(x+256),y:(y+256)]

        tensor_transform = transforms.Compose([
            transforms.Normalize((0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5), (0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5))
        ])
        tensor_image = tensor_transform(tensor_image)

        resize = transforms.Resize((256, 256))  # リサイズするサイズを指定
        tensor_image = resize(tensor_image)  # tensor_imageをリサイズ
        jpg_image = resize(jpg_image)      # jpg_imageをリサイズ


        return {'A': tensor_image, 'B': jpg_image, 'A_paths': tensor_path, 'B_paths': jpg_path}

    def __len__(self):
        return len(self.image_pairs)  # 画像ペアの数を返す

In [5]:
# 生成器Gのクラス定義
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        # U-NetのEocoder部分
        self.down0 = nn.Conv2d(10, 64, kernel_size=4, stride=2, padding=1)
        self.down1 = self.__encoder_block(64, 128)
        self.down2 = self.__encoder_block(128, 256)
        self.down3 = self.__encoder_block(256, 512)
        self.down4 = self.__encoder_block(512, 512)
        self.down5 = self.__encoder_block(512, 512)
        self.down6 = self.__encoder_block(512, 512)
        self.down7 = self.__encoder_block(512, 512, use_norm=False)
        # U-NetのDecoder部分
        self.up7 = self.__decoder_block(512, 512)
        self.up6 = self.__decoder_block(1024, 512, use_dropout=True)
        self.up5 = self.__decoder_block(1024, 512, use_dropout=True)
        self.up4 = self.__decoder_block(1024, 512, use_dropout=True)
        self.up3 = self.__decoder_block(1024, 256)
        self.up2 = self.__decoder_block(512, 128)
        self.up1 = self.__decoder_block(256, 64)
        # Gの最終出力
        self.up0 = nn.Sequential(
            self.__decoder_block(128, 3, use_norm=False),
            nn.Tanh(),
        )

    def __encoder_block(self, input, output, use_norm=True):
        # LeakyReLU＋Downsampling
        layer = [
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(input, output, kernel_size=4, stride=2, padding=1)
        ]
        # BatchNormalization
        if use_norm:
            layer.append(nn.BatchNorm2d(output))
        return nn.Sequential(*layer)

    def __decoder_block(self, input, output, use_norm=True, use_dropout=False):
        # ReLU＋Upsampling
        layer = [
            nn.ReLU(True),
            nn.ConvTranspose2d(input, output, kernel_size=4, stride=2, padding=1)
        ]
        # BachNormalization
        if use_norm:
            layer.append(nn.BatchNorm2d(output))
        # Dropout
        if use_dropout:
            layer.append(nn.Dropout(0.5))
        return nn.Sequential(*layer)

    def forward(self, x):
        # 偽物画像の生成
        x0 = self.down0(x)
        x1 = self.down1(x0)
        x2 = self.down2(x1)
        x3 = self.down3(x2)
        x4 = self.down4(x3)
        x5 = self.down5(x4)
        x6 = self.down6(x5)
        x7 = self.down7(x6)
        y7 = self.up7(x7)
        # Encoderの出力をDecoderの入力にSkipConnectionで接続
        y6 = self.up6(self.concat(x6, y7))
        y5 = self.up5(self.concat(x5, y6))
        y4 = self.up4(self.concat(x4, y5))
        y3 = self.up3(self.concat(x3, y4))
        y2 = self.up2(self.concat(x2, y3))
        y1 = self.up1(self.concat(x1, y2))
        y0 = self.up0(self.concat(x0, y1))

        return y0

    def concat(self, x, y):
        # 特徴マップの結合
        return torch.cat([x, y], dim=1)

In [6]:
# 識別器Dのクラス定義
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        # 70x70PatchGAN識別器モデルの定義
        # 2つの画像を結合したものが入力となるため、チャネル数は3*2=6となる
        self.model = nn.Sequential(
            nn.Conv2d(13, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),
            self.__layer(64, 128),
            self.__layer(128, 256),
            self.__layer(256, 512, stride=1),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1),
        )

    def __layer(self, input, output, stride=2):
        # DownSampling
        return nn.Sequential(
            nn.Conv2d(input, output, kernel_size=4, stride=stride, padding=1),
            nn.BatchNorm2d(output),
            nn.LeakyReLU(0.2, True)
        )

    def forward(self, x):
        return self.model(x)

In [7]:
# GANのAdversarial損失の定義(Real/Fake識別)
class GANLoss(nn.Module):
    def __init__(self):
        super(GANLoss, self).__init__()

        self.register_buffer('real_label', torch.tensor(1.0))
        self.register_buffer('fake_label', torch.tensor(0.0))
        # Real/Fake識別の損失を、シグモイド＋バイナリクロスエントロピーで計算
        self.loss = nn.BCEWithLogitsLoss()

    def __call__(self, prediction, is_real):
        if is_real:
            target_tensor = self.real_label
        else:
            target_tensor = self.fake_label

        return self.loss(prediction, target_tensor.expand_as(prediction))

In [8]:
# Pix2Pixモデルの定義クラス
# 入力と出力の画像ペア間のマッピングを学習するモデル
class Pix2Pix():
    def __init__(self, config):
        self.config = config

        # 生成器Gのオブジェクト取得とデバイス設定
        self.netG = Generator().to(self.config.device)
        # ネットワークの初期化
        self.netG.apply(self.__weights_init)
        # 生成器Gのモデルファイル読み込み(学習を引き続き行う場合)
        if self.config.path_to_generator != None:
            self.netG.load_state_dict(torch.load(self.config.path_to_generator, map_location=self.config.device_name), strict=False)

        # 識別器Dのオブジェクト取得とデバイス設定
        self.netD = Discriminator().to(self.config.device)
        # Dのネットワーク初期化
        self.netD.apply(self.__weights_init)
        # Dのモデルファイル読み込み(学習を引き続き行う場合)
        if self.config.path_to_discriminator != None:
            self.netD.load_state_dict(torch.load(self.config.path_to_discriminator, map_location=self.config.device_name), strict=False)

        # Optimizerの初期化
        self.optimizerG = optim.Adam(self.netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
        self.optimizerD = optim.Adam(self.netD.parameters(), lr=0.0001, betas=(0.5, 0.999))
        # 目的（損失関数)の設定
        # GAN損失(Adversarial損失)
        self.criterionGAN = GANLoss().to(self.config.device)
        # L1損失
        self.criterionL1 = nn.L1Loss()

        # 学習率のスケジューラ設定
        self.schedulerG = optim.lr_scheduler.LambdaLR(self.optimizerG, self.__modify_learning_rate)
        self.schedulerD = optim.lr_scheduler.LambdaLR(self.optimizerD, self.__modify_learning_rate)

        self.training_start_time = time.time()

        self.writer = SummaryWriter(log_dir=config.log_dir)

    def update_learning_rate(self):
        # 学習率の更新、毎エポック後に呼ばれる
        self.schedulerG.step()
        self.schedulerD.step()

    def __modify_learning_rate(self, epoch):
        # 学習率の計算
        # 指定の開始epochから、指定の減衰率で線形に減衰させる
        if self.config.epochs_lr_decay_start < 0:
            return 1.0

        delta = max(0, epoch - self.config.epochs_lr_decay_start) / float(self.config.epochs_lr_decay)
        return max(0.0, 1.0 - delta)

    def __weights_init(self, m):
        # パラメータ初期値の設定
        classname = m.__class__.__name__
        if classname.find('Conv') != -1:
            m.weight.data.normal_(0.0, 0.02)
        elif classname.find('BatchNorm') != -1:
            m.weight.data.normal_(1.0, 0.02)
            m.bias.data.fill_(0)

    def train(self, data, batches_done):
        # ドメインAのラベル画像とドメインBの正解画像を設定
        self.realA = data['A'].to(self.config.device)
        self.realB = data['B'].to(self.config.device)

        # 生成器Gで画像生成
        fakeB = self.netG(self.realA)

        # Discriminator
        # 条件画像(A)と生成画像(B)を結合
        fakeAB = torch.cat((self.realA, fakeB), dim=1)
        # 識別器Dに生成画像を入力、このときGは更新しないのでdetachして勾配は計算しない
        pred_fake = self.netD(fakeAB.detach())
        # 偽物画像を入力したときの識別器DのGAN損失を算出
        lossD_fake = self.criterionGAN(pred_fake, False)

        # 条件画像(A)と正解画像(B)を結合
        realAB = torch.cat((self.realA, self.realB), dim=1)
        # 識別器Dに正解画像を入力
        pred_real = self.netD(realAB)
        # 正解画像を入力したときの識別器DのGAN損失を算出
        lossD_real = self.criterionGAN(pred_real, True)

        # 偽物画像と正解画像のGAN損失の合計に0.5を掛ける
        lossD = (lossD_fake + lossD_real) * 0.5

        # Dの勾配をゼロに設定
        self.optimizerD.zero_grad()
        # Dの逆伝搬を計算
        lossD.backward()
        # Dの重みを更新
        self.optimizerD.step()

        # Generator
        # 評価フェーズなので勾配は計算しない
        # 識別器Dに生成画像を入力
        with torch.no_grad():
            pred_fake = self.netD(fakeAB)

        # 生成器GのGAN損失を算出
        lossG_GAN = self.criterionGAN(pred_fake, True)
        # 生成器GのL1損失を算出
        lossG_L1 = self.criterionL1(fakeB, self.realB) * self.config.lambda_L1

        # 生成器Gの損失を合計
        lossG = lossG_GAN + lossG_L1

        # Gの勾配をゼロに設定
        self.optimizerG.zero_grad()
        # Gの逆伝搬を計算
        lossG.backward()
        # Gの重みを更新
        self.optimizerG.step()

        # for log
        self.fakeB = fakeB
        self.lossG_GAN = lossG_GAN
        self.lossG_L1 = lossG_L1
        self.lossG = lossG
        self.lossD_real = lossD_real
        self.lossD_fake = lossD_fake
        self.lossD = lossD

        train_info = {
            'epoch': epoch,
            'batch_num': batch_num,
            'lossG_GAN': lossG_GAN.item(),
            'lossG_L1': lossG_L1.item(),
            'lossG': lossG.item(),
            'lossD_real': lossD_real.item(),
            'lossD_fake': lossD_fake.item(),
            'lossD': lossD.item(),
            }

        self.save_loss(train_info, batches_done)

    def train_G(self, data, batches_done):
        # ドメインAのラベル画像とドメインBの正解画像を設定
        self.realA = data['A'].to(self.config.device)
        self.realB = data['B'].to(self.config.device)

        # 生成器Gで画像生成
        fakeB = self.netG(self.realA)

        # 生成器GのL1損失を算出
        lossG_L1 = self.criterionL1(fakeB, self.realB) * self.config.lambda_L1

        # 生成器Gの損失を合計
        lossG = lossG_L1

        # Gの勾配をゼロに設定
        self.optimizerG.zero_grad()
        # Gの逆伝搬を計算
        lossG.backward()
        # Gの重みを更新
        self.optimizerG.step()

        # for log
        self.fakeB = fakeB
        self.lossG_L1 = lossG_L1
        self.lossG = lossG

        train_info = {
            'epoch': epoch,
            'batch_num': batch_num,
            'lossG_L1': lossG_L1.item(),
            'lossG': lossG.item(),
            }

        self.save_loss(train_info, batches_done)

    def save_model(self, epoch):
        # モデルの保存
        output_dir = self.config.output_dir
        torch.save(self.netG.state_dict(), '{}/pix2pix_G_epoch_{}'.format(output_dir, epoch))
        torch.save(self.netD.state_dict(), '{}/pix2pix_D_epoch_{}'.format(output_dir, epoch))

    def save_image(self, epoch):
        # 条件画像、生成画像、正解画像を並べて画像を保存
        output_image = torch.cat([self.fakeB, self.realB], dim=3)
        vutils.save_image(output_image,
                '{}/pix2pix_epoch_{}.png'.format(self.config.output_dir, epoch),
                normalize=True)

        self.writer.add_image('image_epoch{}'.format(epoch), self.fakeB[0], epoch)

    def save_loss(self, train_info, batches_done):
        """
        lossの保存
        """
        for k, v in train_info.items():
            self.writer.add_scalar(k, v, batches_done)

In [9]:
# 出力データ格納用フォルダ
!mkdir -p 'output'

サブディレクトリまたはファイル -p は既に存在します。
処理中にエラーが発生しました: -p
サブディレクトリまたはファイル 'output' は既に存在します。
処理中にエラーが発生しました: 'output'


In [10]:
# パラメータの保存
import json
def save_json(file, save_path, mode):
    with open(param_save_path, mode) as outfile:
        json.dump(file, outfile, indent=4)

In [11]:
from torch.utils.tensorboard import SummaryWriter
log_dir = './logs'
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)

In [12]:
!ls ./logs
!rm -rf ./logs/*

'ls' は、内部コマンドまたは外部コマンド、
操作可能なプログラムまたはバッチ ファイルとして認識されていません。
'rm' は、内部コマンドまたは外部コマンド、
操作可能なプログラムまたはバッチ ファイルとして認識されていません。


In [13]:
from __future__ import absolute_import, division, print_function, unicode_literals

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass

%load_ext tensorboard
%tensorboard --logdir './logs'

In [14]:
class Opts():
    def __init__(self):
        self.epochs = 100
        self.save_data_interval = 10
        self.save_image_interval = 10
        self.log_interval = 20
        self.sample_interval = 10
        self.batch_size = 64
        self.load_size = 286
        self.crop_size = 256
        self.cpu = True
        self.dataroot = 'D:/datasets'
        self.output_dir = 'D:/datasets/output_op100ep2'
        self.log_dir = './logs'
        self.phase = 'train_10k(tensor)op'
        self.lambda_L1 = 20.0
        self.epochs_lr_decay = 0
        self.epochs_lr_decay_start = -1
        self.path_to_generator = None
        self.path_to_discriminator = None
        self.device_name = "cuda:0"
        self.device = torch.device(self.device_name)

    def to_dict(self):
        parameters = {
            'epochs': self.epochs,
            'save_data_interval': self.save_data_interval,
            'save_image_interval': self.save_image_interval,
            'log_interval': self.log_interval,
            'sample_interval': self.sample_interval,
            'batch_size': self.batch_size,
            'load_size': self.load_size,
            'crop_size': self.crop_size,
            'cpu': self.cpu,
            'dataroot': self.dataroot,
            'output_dir': self.output_dir,
            'log_dir': self.log_dir,
            'phase': self.phase,
            'lambda_L1': self.lambda_L1,
            'epochs_lr_decay': self.epochs_lr_decay,
            'epochs_lr_decay_start': self.epochs_lr_decay_start,
            'path_to_generator': self.path_to_generator,
            'path_to_discriminator': self.path_to_discriminator,
            'device_name': self.device_name,
        }
        return parameters

opt = Opts()

param_save_path = os.path.join('D:/datasets/output', 'param.json')
save_json(opt.to_dict(), param_save_path, 'w')

In [15]:
model = Pix2Pix(opt)

In [16]:
dataset = AlignedDataset(opt)

In [17]:
def custom_collate_fn(batch):
    """DataLoaderのための照合関数"""


    # 各データ要素の空のリストを初期化
    As = []
    Bs = []
    A_paths = []
    B_paths = []

    # バッチ内の各サンプルを反復処理
    for sample in batch:
        # 各要素を対応するリストに追加
        As.append(sample['A'])

        Bs.append(sample['B'])
        A_paths.append(sample['A_paths'])
        B_paths.append(sample['B_paths'])

    # 画像のリストをスタックし、メタデータをリストにする
    As = torch.stack(As, dim=0)  # 'A'（条件画像）をスタック
    Bs = torch.stack(Bs, dim=0)  # 'B'（正解画像）をスタック

    # A_pathsとB_pathsはリストのままにする

    # 照合されたデータを辞書として返す
    return {
        'A': As,
        'B': Bs,
        'A_paths': A_paths,
        'B_paths': B_paths
    }

In [18]:
dataloader = DataLoader(dataset, batch_size=opt.batch_size, shuffle=True, collate_fn=custom_collate_fn)

In [19]:
for epoch in range(1, opt.epochs + 1):
    for batch_num, data in enumerate(dataloader):
        batches_done = (epoch - 1) * len(dataloader) + batch_num
        
        model.train(data, batches_done)

        if batch_num % opt.log_interval == 0:
            print("===> Epoch[{}]({}/{}): LossD_real: {:.4f}  LossD_fake: {:.4f} LossG_GAN: {:.4f} LossG_L1: {:.4f}".format(
                epoch, batch_num, len(dataloader), model.lossD_real, model.lossD_fake, model.lossG_GAN, model.lossG_L1))

    if epoch % opt.save_data_interval == 0:
        model.save_model(epoch)

    if epoch % opt.save_image_interval == 0:
        model.save_image(epoch)

    model.update_learning_rate()

===> Epoch[1](0/157): LossD_real: 0.9790  LossD_fake: 0.7348 LossG_GAN: 1.0631 LossG_L1: 12.3194
===> Epoch[1](20/157): LossD_real: 0.7359  LossD_fake: 0.7350 LossG_GAN: 0.7169 LossG_L1: 4.1477
===> Epoch[1](40/157): LossD_real: 0.7058  LossD_fake: 0.7081 LossG_GAN: 0.7085 LossG_L1: 2.8648
===> Epoch[1](60/157): LossD_real: 0.7113  LossD_fake: 0.6925 LossG_GAN: 0.7006 LossG_L1: 2.4497
===> Epoch[1](80/157): LossD_real: 0.7018  LossD_fake: 0.6969 LossG_GAN: 0.6972 LossG_L1: 2.0900
===> Epoch[1](100/157): LossD_real: 0.7038  LossD_fake: 0.6979 LossG_GAN: 0.7027 LossG_L1: 2.1239
===> Epoch[1](120/157): LossD_real: 0.6910  LossD_fake: 0.7092 LossG_GAN: 0.6941 LossG_L1: 2.0233
===> Epoch[1](140/157): LossD_real: 0.7009  LossD_fake: 0.7040 LossG_GAN: 0.6929 LossG_L1: 1.8109
===> Epoch[2](0/157): LossD_real: 0.7248  LossD_fake: 0.6886 LossG_GAN: 0.7152 LossG_L1: 2.1192
===> Epoch[2](20/157): LossD_real: 0.6972  LossD_fake: 0.7028 LossG_GAN: 0.6983 LossG_L1: 1.8280
===> Epoch[2](40/157): LossD